# AiXbio — Notebook 3: SAE Feature Discovery

Uses interPLM pre-trained SAEs on ESM-2-650M to find mechanistic toxin-detection features.
Experiment 4: which SAE features fire selectively on toxins, and do they generalise to redesigns?

In [3]:
pip install wandb

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.2/27.2 MB 59.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 460.9/460.9 KB 78.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.0/472.0 KB 70.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.5/209.5 KB 48.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 KB 21.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 111.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 KB 13.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 KB 40.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install -e .

Defaulting to user installation because normal site-packages is not writeable
Obtaining file:///home/ubuntu
ERROR: file:///home/ubuntu does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
Note: you may need to restart the kernel to use updated packages.


In [1]:
import warnings, json
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from Bio import SeqIO
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BEST_LAYER = 33    # interPLM-esm2-650m has SAEs at layers 4,8,12,16,20,24,28,30,33
                   # Override if your Notebook 2 found a different best layer
Path('results').mkdir(exist_ok=True)
print(f'Device: {DEVICE}  |  SAE layer: {BEST_LAYER}')

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Device: cuda  |  SAE layer: 33


## 1. Load interPLM SAE (ESM-2-650M, layer 24)

In [2]:
from interplm.sae.inference import load_sae_from_hf

# Load SAE for ESM-2-650M at layer 24 (matches our embedding layer)
sae = load_sae_from_hf(plm_model='esm2-650m', plm_layer=BEST_LAYER)
sae = sae.to(DEVICE).eval()

# Inspect architecture
print(f'SAE type: {type(sae).__name__}')
print(f'Parameters: {sum(p.numel() for p in sae.parameters()):,}')

# Get input/feature dimensions from weight shapes
for name, param in sae.named_parameters():
    print(f'  {name}: {tuple(param.shape)}')

Loading configs from /home/ubuntu/.cache/huggingface/hub/models--Elana--InterPLM-esm2-650m/snapshots/5121c4c7f3ad0b5fbe0f3b9a457969192bb9912f/layer_33/config.yaml
Loaded data type: <class 'interplm.train.configs.TrainingRunConfig'>
Data keys: Not a dict
SAE type: ReLUSAE
Parameters: 26,225,920
  bias: (1280,)
  encoder.weight: (10240, 1280)
  encoder.bias: (10240,)
  decoder.weight: (1280, 10240)


## 2. Get SAE Feature Activations from Pre-computed Embeddings

In [3]:
def get_sae_features(embeddings: np.ndarray, batch_size: int = 256) -> np.ndarray:
    """
    Run pre-computed ESM-2 embeddings through the SAE encoder.
    Returns feature activation matrix (n_seq, n_features).
    """
    all_acts = []
    with torch.no_grad():
        for i in range(0, len(embeddings), batch_size):
            batch = torch.tensor(embeddings[i:i+batch_size],
                                  dtype=torch.float32).to(DEVICE)
            # interPLM SAE: encode() returns feature activations (post-ReLU)
            try:
                acts = sae.encode(batch)           # preferred method
            except AttributeError:
                acts = sae(batch)[1]               # fallback: (recon, acts, ...) tuple
            all_acts.append(acts.cpu().float().numpy())
    return np.concatenate(all_acts, axis=0)


# Load pre-computed embeddings (from Notebook 1)
tox_embs  = np.load(f'embeddings/natural_toxins_layer{BEST_LAYER}.npy')
ctrl_embs = np.load(f'embeddings/controls_layer{BEST_LAYER}.npy')
rdsg_embs = np.load(f'embeddings/redesigns_layer{BEST_LAYER}.npy')

print(f'Toxins:   {tox_embs.shape}  Controls: {ctrl_embs.shape}  Redesigns: {rdsg_embs.shape}')

print('Running SAE encoder on natural toxins...')
tox_acts  = get_sae_features(tox_embs)
print('Running SAE encoder on controls...')
ctrl_acts = get_sae_features(ctrl_embs)
print('Running SAE encoder on redesigns...')
rdsg_acts = get_sae_features(rdsg_embs)

n_features = tox_acts.shape[1]
print(f'\nFeature matrix shapes:')
print(f'  Toxins:   {tox_acts.shape}')
print(f'  Controls: {ctrl_acts.shape}')
print(f'  Redesigns:{rdsg_acts.shape}')
print(f'  Total features: {n_features:,}')

Toxins:   (1712, 1280)  Controls: (2072, 1280)  Redesigns: (643, 1280)
Running SAE encoder on natural toxins...
Running SAE encoder on controls...
Running SAE encoder on redesigns...

Feature matrix shapes:
  Toxins:   (1712, 10240)
  Controls: (2072, 10240)
  Redesigns:(643, 10240)
  Total features: 10,240


## 3. Identify Toxin-Discriminating Features

In [4]:
# Train/test split on natural sequences
X_nat = np.concatenate([tox_acts, ctrl_acts])
y_nat = np.concatenate([np.ones(len(tox_acts)), np.zeros(len(ctrl_acts))])
idx = np.arange(len(X_nat))
tr_idx, te_idx = train_test_split(idx, test_size=0.2, stratify=y_nat, random_state=42)

X_tr, y_tr = X_nat[tr_idx], y_nat[tr_idx]
X_te, y_te = X_nat[te_idx], y_nat[te_idx]

# Per-feature AUROC on training set
print(f'Computing per-feature AUROC over {n_features:,} features...')
feat_aurocs = np.zeros(n_features)
feat_active = np.zeros(n_features)  # fraction of sequences where feature fires

for f in range(n_features):
    scores = X_tr[:, f]
    if scores.max() > 0 and len(np.unique(scores)) > 1:
        try:
            feat_aurocs[f] = roc_auc_score(y_tr, scores)
        except:
            feat_aurocs[f] = 0.5
    else:
        feat_aurocs[f] = 0.5
    feat_active[f] = (scores > 0).mean()

# Reflect AUROC below 0.5 (anti-toxin features are also informative)
feat_aurocs_abs = np.where(feat_aurocs >= 0.5, feat_aurocs, 1 - feat_aurocs)

# Top-K toxin features
TOP_K = 50
top_feat_idx = np.argsort(feat_aurocs_abs)[::-1][:TOP_K]

print(f'\nTop {TOP_K} toxin features:')
print(f'  AUROC range:    {feat_aurocs_abs[top_feat_idx[0]]:.4f} – {feat_aurocs_abs[top_feat_idx[-1]]:.4f}')
print(f'  Activation rate:{feat_active[top_feat_idx].mean()*100:.1f}% of training sequences')
print(f'  Dead features:  {(feat_aurocs == 0.5).sum():,}/{n_features:,}')

# Save feature stats
feat_stats = {
    'top_feature_indices': top_feat_idx.tolist(),
    'top_feature_aurocs':  feat_aurocs_abs[top_feat_idx].tolist(),
    'top_feature_activation_rates': feat_active[top_feat_idx].tolist(),
    'all_aurocs_mean': float(feat_aurocs_abs.mean()),
    'n_features': int(n_features),
    'n_active_features': int((feat_active > 0).sum()),
    'top_k': TOP_K,
}
with open('results/sae_feature_stats.json', 'w') as f:
    json.dump(feat_stats, f, indent=2)
print('Saved → results/sae_feature_stats.json')

Computing per-feature AUROC over 10,240 features...

Top 50 toxin features:
  AUROC range:    0.6940 – 0.5437
  Activation rate:9.1% of training sequences
  Dead features:  8,345/10,240
Saved → results/sae_feature_stats.json


## 4. SAE-Probe: Classify Using Top-K Features Only

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# ── Full embedding probe (baseline) ──────────────────────────
sc_full = StandardScaler()
X_tr_full = sc_full.fit_transform(X_tr)
X_te_full = sc_full.transform(X_te)
lr_full = LogisticRegression(C=1.0, max_iter=500).fit(X_tr_full, y_tr)
auroc_full = roc_auc_score(y_te, lr_full.predict_proba(X_te_full)[:, 1])

# ── Top-K SAE features probe ──────────────────────────────────
sc_sae = StandardScaler()
X_tr_sae = sc_sae.fit_transform(X_tr[:, top_feat_idx])
X_te_sae = sc_sae.transform(X_te[:, top_feat_idx])
lr_sae = LogisticRegression(C=1.0, max_iter=500).fit(X_tr_sae, y_tr)
auroc_sae = roc_auc_score(y_te, lr_sae.predict_proba(X_te_sae)[:, 1])

print(f'AUROC comparison (natural test set):')
print(f'  Full embedding ({n_features:,} features): {auroc_full:.4f}')
print(f'  Top-{TOP_K} SAE features:              {auroc_sae:.4f}')
print(f'  Compression ratio: {n_features/TOP_K:.0f}× fewer features')

AUROC comparison (natural test set):
  Full embedding (10,240 features): 0.9585
  Top-50 SAE features:              0.9447
  Compression ratio: 205× fewer features


## 5. Generalisation Curve — SAE Features vs Probe vs BLAST

In [7]:
# ── CELL 5 FIX — run this instead of the original Cell 5 ───────────────
seq_identities   = np.load('embeddings/sequence_identities.npy')
blast_identities = np.load('embeddings/blast_identities.npy')

# FIX: X_te_sae is (n_test, 50) — correct. X_te was (n_test, 10240) — wrong.
y_test_all      = np.concatenate([y_te, np.ones(len(rdsg_acts))])
X_test_all_sae  = np.concatenate([X_te_sae,
                                   sc_sae.transform(rdsg_acts[:, top_feat_idx])])
X_test_all_full = np.concatenate([X_te_full,
                                   sc_full.transform(rdsg_acts)])

print(f'X_test_all_sae:  {X_test_all_sae.shape}   ← should be (n, 50)')
print(f'X_test_all_full: {X_test_all_full.shape}  ← should be (n, 10240)')
print(f'y_test_all:      {y_test_all.shape}')

# ── BLAST vs ESM-2 headline numbers ──────────────────────────
rdsg_probe_probs = lr_full.predict_proba(sc_full.transform(rdsg_acts))[:, 1]
esm2_detected    = float((rdsg_probe_probs > 0.5).mean())

print('\n=== BLAST vs ESM-2 on Redesigns ===')
print(f'{"Attack":<35} {"Detected":>10}')
print('-' * 47)
for thr in [0.30, 0.40, 0.50]:
    blast_det = float((blast_identities[len(y_te):] >= thr).mean())
    print(f'  BLAST threshold {thr:.0%}             {blast_det:>10.1%}')
print(f'  ESM-2 probe (top-50 SAE features) {esm2_detected:>10.1%}')


X_test_all_sae:  (1400, 50)   ← should be (n, 50)
X_test_all_full: (1400, 10240)  ← should be (n, 10240)
y_test_all:      (1400,)

=== BLAST vs ESM-2 on Redesigns ===
Attack                                Detected
-----------------------------------------------
  BLAST threshold 30%                   0.0%
  BLAST threshold 40%                   0.0%
  BLAST threshold 50%                   0.0%
  ESM-2 probe (top-50 SAE features)      86.0%


## 6. Feature Robustness Analysis

In [8]:
# Per-feature: does the feature transfer to redesigns?
# Compare activation rate on toxins vs redesigns
print('Feature transfer analysis (top 20 features):')
print(f'{"Feat":>6}  {"Train AUROC":>11}  {"Tox act%":>9}  {"Ctrl act%":>9}  {"Rdsg act%":>9}  {"Transfer":>9}')
print('-'*62)

transfer_results = []
for rank, f in enumerate(top_feat_idx[:20]):
    tox_rate  = (tox_acts[:, f]  > 0).mean()
    ctrl_rate = (ctrl_acts[:, f] > 0).mean()
    rdsg_rate = (rdsg_acts[:, f] > 0).mean()
    # Transfer: does redesign activation ≈ toxin activation?
    transfer = rdsg_rate / (tox_rate + 1e-6)
    auroc_f = feat_aurocs_abs[f]
    print(f"{f:>6}  {auroc_f:>11.4f}  {tox_rate*100:>8.1f}%  {ctrl_rate*100:>8.1f}%  {rdsg_rate*100:>8.1f}%  {transfer:>9.2f}")
    transfer_results.append({'feature': int(f), 'rank': rank+1,
                              'train_auroc': float(auroc_f),
                              'tox_activation': float(tox_rate),
                              'ctrl_activation': float(ctrl_rate),
                              'redesign_activation': float(rdsg_rate),
                              'transfer_ratio': float(transfer)})

mean_transfer = np.mean([r['transfer_ratio'] for r in transfer_results])
print(f'\nMean transfer ratio: {mean_transfer:.2f}')
print('  > 0.8 = feature generalises to redesigns (structurally robust)')
print('  < 0.3 = feature is sequence-identity dependent (evadable)')

Feature transfer analysis (top 20 features):
  Feat  Train AUROC   Tox act%  Ctrl act%  Rdsg act%   Transfer
--------------------------------------------------------------
  6122       0.6940      41.3%       3.8%      99.5%       2.41
  4097       0.6694      37.3%       4.0%      98.6%       2.64
  5312       0.6691      34.8%       0.2%       4.7%       0.13
  1055       0.6438      29.6%       0.9%      99.2%       3.36
  9026       0.6278      28.7%       3.2%       2.0%       0.07
  4397       0.6271       0.4%      26.4%       0.0%       0.00
  9927       0.6173      25.1%       0.7%      24.1%       0.96
  6971       0.6141      25.3%       2.5%      55.4%       2.19
  2704       0.6104       0.2%      22.3%       0.0%       0.00
  1974       0.6097       0.1%      21.7%       0.0%       0.00
  3130       0.6054      21.5%       0.2%       2.8%       0.13
   814       0.6054      22.0%       0.7%      13.1%       0.59
  9487       0.6021      23.1%       2.4%      72.6%       3

## 7. Save All Results

In [9]:
# Load main_results if it exists (from Notebook 2)
try:
    with open('results/main_results.json') as f:
        main_results = json.load(f)
    has_main = True
except FileNotFoundError:
    main_results = {}; has_main = False

sae_results = {
    'sae_model': f'interPLM-esm2-650m-layer{BEST_LAYER}',
    'n_features_total': int(n_features),
    'n_active_features': int((feat_active > 0).sum()),
    'top_k': TOP_K,
    'auroc_full_embedding': float(auroc_full),
    'auroc_sae_top_k': float(auroc_sae),
    'compression_ratio': float(n_features / TOP_K),
    'mean_transfer_ratio': float(mean_transfer),
    'feature_stats': feat_stats,
    'sae_generalisation_curve': sae_curve,
    'feature_transfer': transfer_results,
}

def _conv(o):
    if isinstance(o, (np.integer, np.floating)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return o

with open('results/sae_results.json', 'w') as f:
    json.dump(sae_results, f, default=_conv, indent=2)

# Merge into main_results for Notebook 4
main_results['sae'] = sae_results
with open('results/main_results.json', 'w') as f:
    json.dump(main_results, f, default=_conv, indent=2)

print('Saved → results/sae_results.json')
print('Merged → results/main_results.json')
print()
print('=== SAE Summary ===')
print(f'  Active features: {sae_results["n_active_features"]:,}/{n_features:,}')
print(f'  Full AUROC:      {auroc_full:.4f}')
print(f'  SAE-top-{TOP_K}:    {auroc_sae:.4f}  ({n_features//TOP_K}× compression)')
print(f'  Transfer ratio:  {mean_transfer:.2f} (1.0 = perfect generalisation)')
print(f'  → Ready for Notebook 4 figures')

NameError: name 'sae_curve' is not defined

In [10]:
# ── Recovery: rebuild sae_curve then save ──────────────────────
from sklearn.metrics import roc_auc_score

IDENTITY_BINS = [
    (0.90,1.00,'90–100%'),(0.70,0.90,'70–90%'),(0.50,0.70,'50–70%'),
    (0.30,0.50,'30–50%'),(0.10,0.30,'10–30%'),(0.00,0.10,'<10%'),
]

def eval_bin_classifier(clf, X_all, y_all, mask):
    if mask.sum() < 5 or len(np.unique(y_all[mask])) < 2:
        return float('nan')
    return roc_auc_score(y_all[mask], clf.predict_proba(X_all[mask])[:, 1])

def blast_sens(mask, thr=0.30):
    if mask.sum() < 5: return float('nan')
    y = y_test_all[mask]; pred = (blast_identities[mask] >= thr).astype(int)
    tp = ((pred==1)&(y==1)).sum(); fn = ((pred==0)&(y==1)).sum()
    return float(tp/(tp+fn)) if (tp+fn)>0 else float('nan')

sae_curve = []
print(f'{"Bin":10s}  {"n":>5}  {"SAE probe":>10}  {"Full probe":>10}  {"BLAST":>8}')
print('-'*52)
for lo, hi, label in IDENTITY_BINS:
    mask = (seq_identities >= lo) & (seq_identities < hi)
    auroc_s = eval_bin_classifier(lr_sae,  X_test_all_sae,  y_test_all, mask)
    auroc_f = eval_bin_classifier(lr_full, X_test_all_full, y_test_all, mask)
    bl      = blast_sens(mask)
    sae_curve.append({'bin': label, 'lo': lo, 'hi': hi,
                       'n': int(mask.sum()),
                       'sae_auroc': auroc_s, 'full_auroc': auroc_f,
                       'blast_sensitivity': float(bl)})
    print(f"{label:10s}  {int(mask.sum()):>5}  {auroc_s:>10.3f}  {auroc_f:>10.3f}  {bl:>8.3f}")

# ── Now save everything ─────────────────────────────────────────
def _conv(o):
    if isinstance(o,(np.integer,np.floating)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return o

try:
    with open('results/main_results.json') as f:
        main_results = json.load(f)
except FileNotFoundError:
    main_results = {}

sae_results = {
    'sae_model': 'interPLM-esm2-650m-layer33',
    'n_features_total': int(n_features),
    'n_active_features': int((feat_active > 0).sum()),
    'top_k': TOP_K,
    'auroc_full_embedding': float(auroc_full),
    'auroc_sae_top_k':     float(auroc_sae),
    'compression_ratio':   float(n_features / TOP_K),
    'mean_transfer_ratio': float(mean_transfer),
    'feature_stats':       feat_stats,
    'sae_generalisation_curve': sae_curve,
    'feature_transfer':    transfer_results,
    'blast_vs_esm2': {
        'blast_30pct': 0.0, 'blast_40pct': 0.0, 'blast_50pct': 0.0,
        'esm2_sae50': float(esm2_detected),
    },
}

main_results['sae'] = sae_results
with open('results/sae_results.json', 'w') as f:
    json.dump(sae_results, f, default=_conv, indent=2)
with open('results/main_results.json', 'w') as f:
    json.dump(main_results, f, default=_conv, indent=2)

print('\nSaved → results/sae_results.json + main_results.json')
print(f'Mean transfer ratio: {mean_transfer:.2f}')
print(f'BLAST 0% vs ESM-2 {esm2_detected:.1%} — ready for figures')


Bin             n   SAE probe  Full probe     BLAST
----------------------------------------------------
90–100%         0         nan         nan       nan
70–90%          8         nan         nan     0.250
50–70%         37         nan         nan     0.378
30–50%       1346       0.976       0.904     0.150
10–30%          9         nan         nan     0.000
<10%            0         nan         nan       nan

Saved → results/sae_results.json + main_results.json
Mean transfer ratio: 1.28
BLAST 0% vs ESM-2 86.0% — ready for figures
